# CorroborAI — Rapport d'analyse

**Assistant d'investigation pour la corroboration Système A — RH ↔ Système B — Temps** (défi Loto-Québec *CorroborIA*).

Ce notebook est le rapport destiné au jury. **Toute la logique vient du package `src/`**, le même moteur que
l'application Streamlit : le notebook ne réimplémente aucune règle, il appelle le moteur et commente ses résultats.
Les fichiers officiels de `data/` sont ouverts **en lecture seule**.

## 1. Problème

La corroboration existante indique seulement si un champ est *identique* ou *différent*. Or une différence n'est pas
forcément une erreur : le mapping concatène des valeurs, traduit des codes, déduit des statuts de tables de référence
et calcule des dates à partir de l'historique du poste.

**Objectif :** reconnaître automatiquement les écarts justifiés, isoler les vraies anomalies, soumettre à l'humain
uniquement ce qui reste ambigu, et justifier chaque verdict de façon traçable.

## 2. Méthodologie en trois niveaux

| Niveau | Rôle | Module |
|---|---|---|
| 1. Comparaison brute / normalisation | identique, ou identique après normalisation (dates, booléens, identifiants, espaces, encodage) | `normalize.py` |
| 2. Règles métier déterministes | règles lues dans `Mapping.xlsx` (+ situation d'emploi, motifs, détail du poste) ; appariement transparent des affectations | `mapping.py`, `rules.py`, `matching.py` |
| 3. Analyse assistée | **après** les preuves déterministes : motifs récurrents, problèmes systémiques, valeurs interverties, priorité, confiance, explication | `ai_assist.py` |
| Validation humaine | décisions stockées séparément, verdict moteur immuable | `decisions.py` |

Quatre verdicts : **CONFORME**, **ECART_JUSTIFIE**, **ANOMALIE**, **A_INVESTIGUER**. Chaque cas conserve la règle
appliquée, le niveau qui a tranché, les valeurs brutes et normalisées, la preuve (fichier et ligne) et une explication.

In [1]:
import sys, json, tempfile, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

from src import config as C
from src import presentation as P
from src.load_data import load_all, dataset_overview
from src.mapping import mapping_table
from src.corroboration import Corroborator
from src.normalize import compare_values

## 3. Données

In [2]:
ds = load_all(ROOT / "data")
dataset_overview(ds)

,role,fichier,lignes,colonnes,sha256
0,source,Employe_Source_Anonymise_VF.xlsx,23.0,33.0,c0fa67c60cff
1,destination,Employe_Destination_Anonymise_VF.xlsx,22.0,79.0,487e8ac83839
2,detail,détail_du_poste.xlsx,114.0,12.0,71a9222ca3b5
3,motif,Motif de la situation d'emploi.xlsx,92.0,3.0,5d143df7ee15
4,mapping,Mapping.xlsx,NaN,NaN,dbbfe4106b8c


Le classeur **détail du poste** stocke chaque enregistrement comme une chaîne CSV dans la colonne A ; le chargeur le
détecte, le découpe et convertit les dates Excel sérielles (ex. `37790` → `2003-06-18`).

In [3]:
ds.job_detail[["_excel_row", "IdentifiantPoste", "IdentifiantEmploi", "CodeDirectionAffectée",
               "DateEffetAffectation", "DateEffet", "HeuresSemaineContrat", "HeuresJourContrat"]].head(6)

,_excel_row,IdentifiantPoste,IdentifiantEmploi,CodeDirectionAffectée,DateEffetAffectation,DateEffet,HeuresSemaineContrat,HeuresJourContrat
0,2,35342,6585,397,23604,1964-08-15,40,8
1,3,45985,6585,397,26946,1973-10-09,40,8
2,4,31109,6203,348,37790,2003-06-18,40,8
3,5,31109,6203,348,38051,2004-03-05,40,8
4,6,31109,6203,348,39143,2007-03-02,40,8
5,7,31109,6203,348,40597,2011-02-23,40,8


In [4]:
print(f"Source : {len(ds.source)} affectations, {ds.source['Matricule'].nunique()} employés")
print(f"Destination : {len(ds.destination)} lignes, {ds.destination['personId'].nunique()} employés")
print("Types d'affectation source :", ds.source["TypeAffectation"].value_counts().to_dict())

Source : 23 affectations, 20 employés
Destination : 22 lignes, 20 employés
Types d'affectation source : {'P': 20, 'S': 2, 'A': 1}


## 4. Mapping et normalisation

Seuls les champs listés dans `Mapping.xlsx` sont corroborés (consigne du défi). Le mapping est lu directement,
cellules fusionnées comprises, puis converti en représentation interne : *champ(s) source → champ destination →
règle → preuve*.

In [5]:
cor = Corroborator(ds)
mapping_table(cor.engine.entries)[["entry_id", "champs_source", "champs_destination", "type", "corrobore", "preuve"]]

,entry_id,champs_source,champs_destination,type,corrobore,preuve
0,M02,PrénomUsuel,givenName,directe (N/A),oui,"Mapping.xlsx › Mapping, ligne 2"
1,M03,NomFamille,surname,directe (N/A),oui,"Mapping.xlsx › Mapping, ligne 3"
2,M04,Adresse courriel travail,contactEmail,transformation,oui,"Mapping.xlsx › Mapping, lignes 4-6"
3,M07,DateEmbaucheRécente,onboardDate,directe (N/A),oui,"Mapping.xlsx › Mapping, ligne 7"
4,M08,Matricule,personId,directe (N/A),cle d'appariement,"Mapping.xlsx › Mapping, ligne 8"
5,M09,LibelléSite,siteName,directe (N/A),oui,"Mapping.xlsx › Mapping, ligne 9"
6,M10,CodeSite,siteCode,directe (N/A),oui,"Mapping.xlsx › Mapping, ligne 10"
7,M11,CodeDirection,divisionId,directe (N/A),oui,"Mapping.xlsx › Mapping, ligne 11"
8,M12,LibelléDirection,divisionName,transformation,oui,"Mapping.xlsx › Mapping, ligne 12"
9,M13,CodeImputation,divisionCode,directe (N/A),oui,"Mapping.xlsx › Mapping, ligne 13"


In [6]:
print("Champs destination corroborés :", len(cor.engine.field_specs))
pd.DataFrame([{**r.conditions, "→ contractTypeCode": r.result} for r in cor.engine.contract_rules])

Champs destination corroborés : 24


,EstPermanent,EstTempsPlein,CatégorieEmploi,→ contractTypeCode
0,1,1,V,JWN
1,1,0,V,XFLR
2,NaN,NaN,T,KELH
3,NaN,NaN,O,WHX
4,NaN,NaN,M,CEGQ
5,NaN,NaN,R,CNZC
6,NaN,NaN,J,RMQ
7,NaN,NaN,Z,JAW
8,NaN,NaN,Q,TRSY


Exemples de normalisation (niveau 1) — la valeur brute est toujours conservée :

In [7]:
from datetime import datetime
examples = [(datetime(1995, 2, 9), "1995-02-09T00:00:00.000Z", "date"), (397, "00397", "id"),
            ("Oui", "true", "bool"), ("7.2", 7.2, "number"), (" Pre1 ", "PRE1", "name"),
            ("Absence complète", "Absence complÃ¨te", "text"), (35, 40, "number")]
pd.DataFrame([{"source": repr(a), "destination": repr(b), "type": k, **compare_values(a, b, k)} for a, b, k in examples])

,source,destination,type,stage1,src_norm,dst_norm
0,"datetime.datetime(1995, 2, 9, 0, 0)",'1995-02-09T00:00:00.000Z',date,IDENTIQUE_APRES_NORMALISATION,1995-02-09,1995-02-09
1,397,'00397',id,IDENTIQUE_APRES_NORMALISATION,397,397
2,'Oui','true',bool,IDENTIQUE_APRES_NORMALISATION,True,True
3,'7.2',7.2,number,IDENTIQUE,7.2,7.2
4,' Pre1 ','PRE1',name,IDENTIQUE_APRES_NORMALISATION,pre1,pre1
5,'Absence complète','Absence complÃ¨te',text,IDENTIQUE_APRES_NORMALISATION,absence complète,absence complète
6,35,40,number,DIFFERENT,35,40


## 5. Appariement des affectations

La destination ne porte pas le numéro de poste. Une affectation source est donc appariée à une ligne destination de
la même personne grâce à des preuves pondérées : code emploi, indicateurs primaire/temporaire, date d'effet,
direction, site, échelle. Toutes les solutions optimales sont énumérées ; si plusieurs sont équivalentes,
l'appariement est **ambigu** et n'est jamais forcé.

In [8]:
result = cor.run()      # exécution complète des 3 niveaux, sans décision humaine (état de référence)
cases = result.cases
result.matches["statut_appariement"].value_counts().rename("affectations").to_frame()

,affectations
statut_appariement,
APPARIE,22
SOURCE_SEULEMENT,1


In [9]:
result.matches[result.matches["person_id"] == "1545850"][["statut_appariement", "type_affectation", "code_poste",
                                                          "code_emploi_source", "score", "note"]]

,statut_appariement,type_affectation,code_poste,code_emploi_source,score,note
0,APPARIE,P,35342,6585,10.0,plusieurs affectations source candidates pour cette ligne destination ; départagées par les preuves second...
1,SOURCE_SEULEMENT,A,45985,6585,0.0,"aucune ligne destination correspondante ; la ligne destination la plus proche (ligne 2, score 6) est déjà ..."


**Employé 1545850 :** l'affectation principale P est retrouvée dans Temps, mais l'affectation temporaire A (même code
emploi 6585) n'a aucune correspondance claire. Ce cas est détaillé en section 9.

## 6. Comparaison brute

Une corroboration purement brute signalerait chaque différence comme un problème potentiel. La décomposition
ci-dessous est une **partition exacte** de tous les contrôles.

In [10]:
f = P.funnel(cases)
print(f"{f['total']} contrôles = {f['identical']} identiques dès la comparaison + {f['raw_diff']} différences brutes")
print(f"{f['raw_diff']} différences brutes = {f['normalized']} normalisées (format) + {f['derived_conform']} valeurs "
      f"dérivées conformes + {f['justified']} écarts justifiés + {f['anomalies']} anomalies + "
      f"{f['investigate']} à investiguer")
print("Réconciliation :", f["check_total"] and f["check_diff"])

529 contrôles = 317 identiques dès la comparaison + 212 différences brutes
212 différences brutes = 22 normalisées (format) + 20 valeurs dérivées conformes + 111 écarts justifiés + 56 anomalies + 3 à investiguer
Réconciliation : True


In [11]:
pd.crosstab(cases["stage1_result"], cases["verdict"]).reindex(columns=C.VERDICTS, fill_value=0)

verdict,CONFORME,ECART_JUSTIFIE,ANOMALIE,A_INVESTIGUER
stage1_result,,,,
DESTINATION_VIDE,0,20,0,0
DIFFERENT,0,89,34,0
IDENTIQUE,253,0,0,0
IDENTIQUE_APRES_NORMALISATION,22,0,0,0
NON_COMPARABLE,20,2,22,1
SOURCE_VIDE,0,0,0,2
VIDE_DES_DEUX_COTES,64,0,0,0


## 7. Règles métier déterministes

Le niveau 2 applique les règles du mapping et explique la majorité des différences brutes.

In [12]:
pd.crosstab(cases["field"], cases["verdict"]).reindex(columns=C.VERDICTS, fill_value=0)

verdict,CONFORME,ECART_JUSTIFIE,ANOMALIE,A_INVESTIGUER
field,,,,
__assignment__,0,0,0,1
assignmentEndDate,22,0,0,0
assignmentStartDate,19,3,0,0
contactEmail,0,0,22,0
contractTypeCode,0,18,4,0
dailyHoursOverride,18,0,3,1
detailedStatus,20,2,0,0
divisionCode,22,0,0,0
divisionId,22,0,0,0


In [13]:
def show(person, field):
    c = cases[(cases["person_id"] == str(person)) & (cases["field"] == field)].iloc[0]
    print(f"[{c['verdict']}] {field} — employé {person}")
    print("  source :", c["source_value"], "| destination :", c["destination_value"], "| attendu :", c["expected_value"])
    print("  explication :", c["explanation"])
    print()

show(1545850, "siteCode")             # conforme : comparaison directe
show(1545850, "contractTypeCode")     # écart justifié : V / permanent / temps plein → JWN
show(7603160, "statusReasonCode")     # écart justifié : motif 807 → code Remphor 170
show(2911996, "weeklyHoursOverride")  # anomalie : 35 h dans RH, 40 h dans Temps

[CONFORME] siteCode — employé 1545850
  source : 48 | destination : 48 | attendu : 48
  explication : Niveau 1 : valeurs identiques (CodeSite='48', siteCode='48').

[ECART_JUSTIFIE] contractTypeCode — employé 1545850
  source : V | destination : JWN | attendu : JWN
  explication : Niveau 1 : CatégorieEmploi='V' ≠ contractTypeCode='JWN'. Niveau 2 : écart expliqué par R_CONTRACT_TYPE [Mapping.xlsx › Mapping, lignes 24-40] : ligne de règle appliquée : SI PERM_IND=1 et FT_IND=1 et EMPTP_CD="V" --> Mettre "JWN" → attendu 'JWN'.

[ECART_JUSTIFIE] statusReasonCode — employé 7603160
  source : 807 | destination : 170 | attendu : 170
  explication : Niveau 1 : CodeRaisonStatut='807' ≠ statusReasonCode='170'. Niveau 2 : écart expliqué par R_SITUATION_EMPLOI [Mapping.xlsx › Mapping, lignes 17-21] : situation 'Absence complète' (codes d'accès [2, 3, 6, 7], ligne 3) → code Remphor du motif 807 = 170 → attendu '170'.

[ANOMALIE] weeklyHoursOverride — employé 2911996
  source : 35 | destination : 40 

### « Différent » n'est pas « erreur » : `assignmentStartDate`

La source ne contient que la date d'effet du poste. La destination applique la transformation documentée avec
l'historique du détail du poste : `MAX(DateEntréePoste, date d'effet de l'enregistrement courant du détail du poste)`.
Pour 9989151, 3241002 et 4402456, la date brute diffère de la cible, mais la valeur transformée attendue correspond
exactement à la destination : **ECART_JUSTIFIE**. La règle reproduit 22/22 dates destination (clarification
officielle du 2026-10-04).

In [14]:
c = cases[(cases["field"] == "assignmentStartDate") & (cases["stage1_result"] == "DIFFERENT")]
audit = c[["person_id", "source_value", "expected_value", "destination_value", "verdict"]].copy()
audit.columns = ["employé", "date source (RH)", "valeur transformée attendue", "destination (Temps)", "verdict"]
audit["enregistrement courant du détail du poste"] = c["context"].map(lambda x: json.loads(x)["methode_detail_poste"])
audit

,employé,date source (RH),valeur transformée attendue,destination (Temps),verdict,enregistrement courant du détail du poste
163,3241002,2009-03-30,2022-10-05,2022-10-05,ECART_JUSTIFIE,"enregistrement courant (ligne 68) différent du précédent (ligne 67) sur : CodePosteSecondaire, MatriculeGe..."
211,4402456,2001-02-05,2013-10-20,2013-10-20,ECART_JUSTIFIE,"enregistrement courant (ligne 80) différent du précédent (ligne 79) sur : CodePosteSecondaire, MatriculeGe..."
523,9989151,2009-03-30,2021-03-30,2021-03-30,ECART_JUSTIFIE,"enregistrement courant (ligne 10) différent du précédent (ligne 9) sur : CodePosteSecondaire, MatriculeGes..."


## 8. Résultats sur les données officielles

In [15]:
s = result.summary
w = P.workload(cases, result.patterns.assign(human_decision=""))
print(f"{s['n_persons']} employés · {s['n_source_assignments']} affectations source · "
      f"{s['n_destination_rows']} lignes destination · {s['n_fields_corroborated']} champs corroborés")
print(f"{s['n_cases']} contrôles : CONFORME {s['n_CONFORME']} · ECART_JUSTIFIE {s['n_ECART_JUSTIFIE']} · "
      f"ANOMALIE {s['n_ANOMALIE']} · A_INVESTIGUER {s['n_A_INVESTIGUER']}")
print(f"Décisions humaines requises : {w['required']} = {w['individual']} décisions individuelles + "
      f"{w['groups']} décisions de groupe systémique")

20 employés · 23 affectations source · 22 lignes destination · 24 champs corroborés
529 contrôles : CONFORME 359 · ECART_JUSTIFIE 111 · ANOMALIE 56 · A_INVESTIGUER 3
Décisions humaines requises : 17 = 15 décisions individuelles + 2 décisions de groupe systémique


### Problèmes systémiques : 44 anomalies, 2 décisions

44 cas d'anomalie appartiennent à **2 groupes systémiques** : `contactEmail` (22) et `positionName` (22).
Ils **restent ANOMALIE** dans la piste d'audit. Sur le plan opérationnel, ces **44 enregistrements d'anomalie donnent
2 décisions humaines de groupe**.

In [16]:
sys_groups = result.patterns[result.patterns["pattern_type"] == "SYSTEMIQUE_CHAMP"]
pd.DataFrame({"groupe": sys_groups["title"], "cas": sys_groups["n_cases"], "priorité": sys_groups["priority"],
              "verdict des cas": [", ".join(sorted(set(cases.loc[cases["case_id"].isin(ids.split(";")), "verdict"])))
                                  for ids in sys_groups["case_ids"]]})

,groupe,cas,priorité,verdict des cas
3,Problème systémique — génération des courriels (22/22),22,BASSE,ANOMALIE
4,Transformation positionName incorrecte (22/22),22,HAUTE,ANOMALIE


In [17]:
ind = cases[cases["verdict"].isin(["ANOMALIE", "A_INVESTIGUER"]) & ~cases["systemic"]]
ind[["person_id", "field", "verdict", "source_value", "destination_value", "expected_value", "priority"]].reset_index(drop=True)

,person_id,field,verdict,source_value,destination_value,expected_value,priority
0,1545850,__assignment__,A_INVESTIGUER,A · poste 45985 · emploi 6585 · début 2025-09-22,,ligne destination attendue (voir action proposée),HAUTE
1,2762457,contractTypeCode,ANOMALIE,V,WHX,JWN,HAUTE
2,2911996,weeklyHoursOverride,ANOMALIE,35,40,35,HAUTE
3,2911996,dailyHoursOverride,ANOMALIE,7,8,7,HAUTE
4,3241002,siteName,ANOMALIE,Emplacement48,Emplacement35,Emplacement48,HAUTE
5,3712987,contractTypeCode,ANOMALIE,V,XFLR,JWN,HAUTE
6,3712987,weeklyHoursOverride,A_INVESTIGUER,,40,,HAUTE
7,3712987,dailyHoursOverride,A_INVESTIGUER,,8,,HAUTE
8,4402456,weeklyHoursOverride,ANOMALIE,36,40,36,HAUTE
9,4402456,dailyHoursOverride,ANOMALIE,7.2,8,7.2,HAUTE


Registre des ambiguïtés de règles (AMB-01 à AMB-12) et clarifications officielles :

In [18]:
result.ambiguities[["ambiguity_id", "champ", "statut", "statistiques"]]

,ambiguity_id,champ,statut,statistiques
0,AMB-01,assignmentStartDate,RÉSOLUE (2026-10-04),"transformation reproduite : 22/22 lignes destination (3 différentes de la date source, dont 9989151, 32410..."
1,AMB-02,detailedStatus / statusReasonCode / expectedReturnDate,hypothèse retenue (non confirmée),
2,AMB-03,detailedStatus,hypothèse retenue (non confirmée),
3,AMB-04,detailedStatus,hypothèse retenue (non confirmée),
4,AMB-05,statusReasonCode,hypothèse retenue (non confirmée),
5,AMB-06,Motifs,hypothèse retenue (non confirmée),
6,AMB-07,contractTypeCode,hypothèse retenue (non confirmée),
7,AMB-08,isPrimaryAssignment / isTemporaryAssignment,hypothèse retenue (non confirmée),
8,AMB-09,contactEmail,RÉSOLUE (2026-10-04),
9,AMB-10,divisionName / positionName,hypothèse retenue (non confirmée),


In [19]:
from src.export import export_report, export_csv
path = export_report(result, ROOT / "outputs" / "corroboration_report.xlsx")   # état de référence, 0 décision
export_csv(result, ROOT / "outputs")
print("Rapport :", path.relative_to(ROOT))

Rapport : outputs\corroboration_report.xlsx


## 9. Analyse assistée (contribution de l'IA)

L'analyse assistée s'exécute **après** les preuves déterministes. Elle ne remplace aucun verdict : elle groupe,
explique, priorise et suggère. Les simples comparaisons d'égalité ne sont **pas** de l'IA. Aucun LLM ni API externe
n'est appelé.

### 9.1 Valeurs interverties : 2762457 ↔ 4625374

In [20]:
for pid in ("2762457", "4625374"):
    c = cases[(cases["person_id"] == pid) & (cases["field"] == "contractTypeCode")].iloc[0]
    print(f"{pid} : attendu {c['expected_value']} → reçu {c['destination_value']} ({c['verdict']})")
card = P.ai_card(cases[(cases["person_id"] == "2762457") & (cases["field"] == "contractTypeCode")].iloc[0],
                 cases, result.patterns)
print("\nHypothèse :", card["hypothesis"]); print("Cause possible :", card["cause"]); print("Action :", card["action"])

2762457 : attendu JWN → reçu WHX (ANOMALIE)
4625374 : attendu WHX → reçu JWN (ANOMALIE)

Hypothèse : Les valeurs de deux employés semblent avoir été interverties.
Cause possible : Erreur d'ordre ou de clé lors du chargement des données.
Action : Vérifier les deux enregistrements ensemble avant correction.


### 9.2 Cause commune récurrente : heures = valeur par défaut du poste

In [21]:
rec = result.patterns[result.patterns["pattern_id"] == "P-REC-REPLI_VALEUR_DEFAUT_POSTE"].iloc[0]
m = cases[cases["case_id"].isin(rec["case_ids"].split(";"))]
print(P.clue_info(rec, m)["title"])
m[["person_id", "field", "source_value", "destination_value", "verdict"]].reset_index(drop=True)

Plusieurs écarts d'heures correspondent à la valeur par défaut du poste


,person_id,field,source_value,destination_value,verdict
0,2911996,weeklyHoursOverride,35,40,ANOMALIE
1,2911996,dailyHoursOverride,7,8,ANOMALIE
2,3712987,weeklyHoursOverride,,40,A_INVESTIGUER
3,3712987,dailyHoursOverride,,8,A_INVESTIGUER
4,4402456,weeklyHoursOverride,36,40,ANOMALIE
5,4402456,dailyHoursOverride,7.2,8,ANOMALIE
6,7683990,weeklyHoursOverride,36,40,ANOMALIE
7,7683990,dailyHoursOverride,7.2,8,ANOMALIE


### 9.3 Regroupement systémique

In [22]:
for _, p in sys_groups.iterrows():
    print(f"{p['title']} — {p['n_cases']} cas, priorité {p['priority']}")
    print(f"   hypothèse : {p['hypothesis']}")

Problème systémique — génération des courriels (22/22) — 22 cas, priorité BASSE
   hypothèse : artefact d'anonymisation
Transformation positionName incorrecte (22/22) — 22 cas, priorité HAUTE
   hypothèse : substitution cohérente 1:1 sur 9 valeurs distinctes (ex. 'empl6585' → '3649-empl3649', 'empl6900' → '5123-empl5123', 'empl6972' → '5126-empl5126')


### 9.4 Priorité et confiance (formules transparentes)

In [23]:
top = ind.sort_values("priority_score", ascending=False).head(6)
pd.DataFrame({"employé": top["person_id"], "problème": top.apply(P.problem_title, axis=1),
              "priorité": top["priority_score"], "confiance": top["confidence"].map(P.confidence_text),
              "facteurs de confiance": top.apply(lambda r: " ; ".join(f"{s} {t}" for s, t in P.confidence_factors(r)[0]),
                                                 axis=1)})

,employé,problème,priorité,confiance,facteurs de confiance
112,2762457,Type d'emploi — valeurs interverties,85,Élevée — 95 %,+ Règle métier explicite (Mapping.xlsx) ; + Valeur attendue calculable sans ambiguïté ; + Correspondance R...
141,2911996,Heures par semaine incohérentes,85,Élevée — 95 %,+ Comparaison directe des deux valeurs (copie attendue) ; + Valeur attendue calculable sans ambiguïté ; + ...
142,2911996,Heures par jour incohérentes,85,Élevée — 95 %,+ Comparaison directe des deux valeurs (copie attendue) ; + Valeur attendue calculable sans ambiguïté ; + ...
184,3712987,Type d'emploi — valeurs interverties,85,Élevée — 95 %,+ Règle métier explicite (Mapping.xlsx) ; + Valeur attendue calculable sans ambiguïté ; + Correspondance R...
232,4625374,Type d'emploi — valeurs interverties,85,Élevée — 95 %,+ Règle métier explicite (Mapping.xlsx) ; + Valeur attendue calculable sans ambiguïté ; + Correspondance R...
214,4402456,Heures par jour incohérentes,85,Élevée — 95 %,+ Comparaison directe des deux valeurs (copie attendue) ; + Valeur attendue calculable sans ambiguïté ; + ...


### 9.5 Cas ambigu principal : employé 1545850

* **Source :** affectation principale retrouvée ; affectation temporaire non clairement appariée.
* **Conclusion :** **A_INVESTIGUER**.
* **Raison :** les règles et données disponibles ne permettent pas de décider si l'absence dans Temps est légitime.
* **Action :** une affectation destination candidate est proposée, avec la certitude de chaque valeur, mais une
  validation humaine est requise.

In [24]:
miss = cases[cases["case_type"] == "AFFECTATION_ABSENTE_DESTINATION"].iloc[0]
print(miss["verdict"], "—", P.problem_title(miss))
for sign, text in P.confidence_factors(miss)[0]:
    print(f"   {sign} {text}")
prop = result.proposals[result.proposals["case_id"] == miss["case_id"]]
key = prop[~prop["derivation"].str.startswith("champ non couvert")]
key[["field", "proposed_value", "certainty"]].assign(certitude=key["certainty"].map(P.CERTAINTY_LABEL))

A_INVESTIGUER — Affectation temporaire non retrouvée
   + Une affectation RH ne trouve pas de correspondance dans Temps
   + Les affectations RH sont bien distinctes (poste, type, date)
   − Aucune règle ne précise si ce type d'affectation devait être transféré
   − L'absence dans Temps pourrait être légitime


,field,proposed_value,certainty,certitude
0,personId,1545850,CERTAIN,Certain
1,givenName,Pre1545850,CERTAIN,Certain
2,surname,Nom1545850,CERTAIN,Certain
3,contactEmail,dev-08-v2_PNom1545850850@loto-quebec.com,DERIVE,Dérivé d'une règle
5,onboardDate,1995-02-09,CERTAIN,Certain
18,statusReasonCode,,DERIVE,Dérivé d'une règle
19,expectedReturnDate,,DERIVE,Dérivé d'une règle
49,contractTypeCode,JWN,DERIVE,Dérivé d'une règle
57,detailedStatus,Actif,DERIVE,Dérivé d'une règle
59,siteCode,48,CERTAIN,Certain


### 9.6 Apprentissage à partir des décisions d'experts (`FeedbackModel`)

Ce petit arbre de décision, local et explicable, est entraîné sur des décisions d'experts. Il **suggère** seulement un
verdict pour les cas A_INVESTIGUER. Démonstration sur données synthétiques avec des décisions d'experts **simulées**
(politique cachée) : le modèle apprend sur une moitié des cas et prédit l'autre.

In [25]:
from src.synthetic import generate, evaluate
from src.ai_assist import FeedbackModel
synth_dir = ROOT / "synthetic_data" / "synth_n1000_s42"
if not (synth_dir / "ground_truth.csv").exists():
    generate(1000, seed=42, out_dir=synth_dir)
gt = pd.read_csv(synth_dir / "ground_truth.csv", dtype=str)
sres = Corroborator(load_all(synth_dir)).run()
amb_s = sres.cases[sres.cases["verdict"] == "A_INVESTIGUER"].merge(
    gt[["person_id", "src_poste", "field", "simulated_expert_decision"]], on=["person_id", "src_poste", "field"], how="left")
lab = amb_s[amb_s["simulated_expert_decision"].fillna("") != ""].reset_index(drop=True)
train, test = lab.iloc[::2], lab.iloc[1::2]
model = FeedbackModel().fit(train, train["simulated_expert_decision"])
pred, _ = model.predict(test)
print(f"Cas étiquetés : {len(lab)} (entraînement {len(train)}, test {len(test)})")
print("Exactitude des suggestions sur la moitié test :", round((pred == test["simulated_expert_decision"].values).mean(), 3))
print(model.rules_text())

Cas étiquetés : 53 (entraînement 27, test 26)
Exactitude des suggestions sur la moitié test : 1.0
|--- hypothesis_code=AFFECTATION_MANQUANTE <= 0.50
|   |--- class: ANOMALIE
|--- hypothesis_code=AFFECTATION_MANQUANTE >  0.50
|   |--- src_type=A <= 0.50
|   |   |--- class: ECART_JUSTIFIE
|   |--- src_type=A >  0.50
|   |   |--- class: ANOMALIE



## 10. Validation humaine (concept)

* Le **verdict moteur est immuable** ; la **décision humaine est stockée séparément**, dans un journal append-only.
* Une révision ajoute un nouvel événement : la dernière décision est active et l'historique reste auditable.
* Les corrections acceptées alimentent un plan de corrections exporté séparément. Le fichier officiel n'est jamais
  modifié.

Démonstration avec un **journal temporaire**, ce qui laisse intact l'état de référence de l'application :

In [26]:
from src.decisions import DecisionStore
from src.export import corrected_destination
with tempfile.TemporaryDirectory() as tmp:
    store = DecisionStore(Path(tmp) / "decisions.csv")
    target = cases[(cases["person_id"] == "2762457") & (cases["field"] == "contractTypeCode")].iloc[0]
    store.record("case", target["case_id"], C.DECISION_CONFIRM, "démo", dataset_key=result.dataset_key)
    reviewed = Corroborator(load_all(ROOT / "data")).run(decisions=store)
    r = reviewed.cases.set_index("case_id").loc[target["case_id"]]
    _, changes = corrected_destination(reviewed)
    print("Verdict moteur      :", r["verdict"])
    print("Décision humaine    :", P.decision_label(r, r["human_decision"]))
    print("Correction acceptée :", changes.iloc[0]["avant"], "→", changes.iloc[0]["apres"], "(simulation)")
    print("Fichier officiel    : inchangé —", ds.destination.loc[ds.destination["personId"] == 2762457,
                                                                   "contractTypeCode"].iloc[0])

Verdict moteur      : ANOMALIE
Décision humaine    : Anomalie confirmée
Correction acceptée : WHX → JWN (simulation)
Fichier officiel    : inchangé — WHX


## 11. Validation synthétique

Le générateur ré-implémente **indépendamment** les règles documentées, injecte des cas connus (conformes, écarts
justifiés, valeurs erronées ou interverties, affectations manquantes, défaillance systémique, cas ambigus…) et
enregistre une vérité terrain séparée.

In [27]:
print(f"{sres.summary['n_persons']} employés, {sres.summary['n_source_assignments']} affectations, "
      f"{sres.summary['n_cases']} contrôles, {len(gt)} étiquettes injectées")
ev = evaluate(sres.cases, gt)
print("Exactitude (classes OK / ANOMALIE / A_INVESTIGUER) :", ev["accuracy_class"])
ev["confusion_matrix"]

1000 employés, 1162 affectations, 27037 contrôles, 761 étiquettes injectées
Exactitude (classes OK / ANOMALIE / A_INVESTIGUER) : 1.0


predicted_class,OK,ANOMALIE,A_INVESTIGUER
expected_class,,,
OK,26727,0,0
ANOMALIE,0,199,0
A_INVESTIGUER,0,0,111


In [28]:
ev["per_category"]

,category,n,classe_correcte,verdict_exact,regroupe_en_motif
0,ambiguous_duplicate_assignment,16.0,1.0,1.0,0.0000
1,ambiguous_uncovered_rule,10.0,1.0,1.0,0.0000
2,ambiguous_undocumented_code,48.0,1.0,1.0,0.0000
3,format_only,53.0,1.0,1.0,0.0000
4,justified_rule_active_status,22.0,1.0,1.0,0.0000
5,justified_rule_job_history_update,266.0,1.0,1.0,0.0000
6,justified_rule_unit_change,110.0,1.0,1.0,0.0000
7,missing_assignment,21.0,1.0,1.0,1.0000
8,missing_destination_value,8.0,1.0,1.0,1.0000
9,missing_source_value,16.0,1.0,1.0,1.0000


> **100 % sur la vérité terrain générée démontre une implémentation fidèle des règles et la détection des motifs
> injectés. Cela ne prouve pas une couverture complète des situations RH réelles** : le générateur encode la même
> lecture de la documentation que le moteur.

## 12. Limites et hypothèses

* **Clarifications officielles (2026-10-04) intégrées :**
  * `positionName` : erreur réelle confirmée → ANOMALIE, regroupée en un problème systémique ;
  * `contactEmail` : le préfixe d'environnement (ex. `dev-08-v2_`) est autorisé et ignoré ; l'identifiant
    anonymisé divergent est un artefact d'anonymisation connu, mais il est détecté comme anomalie dans le cadre du
    défi et regroupé ;
  * `assignmentStartDate` : résolu ; la transformation avec le détail du poste reproduit 22/22 dates.
* **Lecture de la règle de date :** le texte du mapping seul est ambigu (« plus ancienne », changement détecté sur le
  seul code d'unité). C'est la clarification officielle qui valide l'interprétation retenue.
* **Anomalies systémiques :** elles restent ANOMALIE dans l'audit ; elles sont seulement regroupées pour le flux de
  décision humaine.
* **Registre des ambiguïtés :** AMB-01 à AMB-12. AMB-01, AMB-09 et AMB-12 sont résolues par clarification ; les autres
  restent des hypothèses documentées, sans règle inventée (code non documenté ou combinaison non couverte →
  A_INVESTIGUER).
* **Échantillon officiel très petit** (20 employés) : les seuils du niveau 3 sont paramétrables dans `src/config.py`.
* **IA :** entièrement locale ; `LLMAssistant` n'est qu'un point d'extension futur (prompts caviardés, aucun appel).

## 13. Conclusion

In [29]:
print(f"{s['n_cases']} contrôles → {f['identical']} identiques + {f['raw_diff']} différences brutes")
print(f"{f['raw_diff']} différences brutes → {s['n_ECART_JUSTIFIE']} écarts justifiés automatiquement, "
      f"{f['normalized'] + f['derived_conform']} conformes après normalisation/dérivation")
print(f"{s['n_ANOMALIE']} anomalies + {s['n_A_INVESTIGUER']} cas ambigus → {w['required']} décisions humaines "
      f"({w['individual']} individuelles + {w['groups']} de groupe)")

529 contrôles → 317 identiques + 212 différences brutes
212 différences brutes → 111 écarts justifiés automatiquement, 42 conformes après normalisation/dérivation
56 anomalies + 3 cas ambigus → 17 décisions humaines (15 individuelles + 2 de groupe)


CorroborAI sépare clairement ce qui relève de la **règle déterministe** (verdict traçable, immuable) de ce qui relève
de l'**analyse assistée** (regroupement, cause probable, priorité, explication). Il réduit la charge d'investigation
de 212 différences brutes à **17 décisions humaines**, sans jamais modifier les fichiers officiels et en gardant
chaque verdict relié à ses données et à sa règle.